# Logistics Operations EDA

Notebook simples para listar os arquivos disponíveis e ler alguns registros do dataset usando o leitor híbrido do projeto.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    for parent in [project_root, *project_root.parents]:
        if (parent / "src").exists():
            project_root = parent
            break

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Project root: {project_root}")

Project root: D:\developer\workspace_python\logistics-operations


In [2]:
import pandas as pd
from logistics_ops.bootstrap import build_tabular_reader

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

reader = build_tabular_reader()

D:\developer\workspace_python\logistics-operations\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset_objects = reader.list_dataset_objects()
print(f"Total de objetos encontrados: {len(dataset_objects)}")
dataset_objects[:10]

Total de objetos encontrados: 16


['raw/kaggle/yogape/logistics-operations-database/.complete/datasets/yogape/logistics-operations-database/1/bundle.complete',
 'raw/kaggle/yogape/logistics-operations-database/DATABASE_SCHEMA.txt',
 'raw/kaggle/yogape/logistics-operations-database/customers.csv',
 'raw/kaggle/yogape/logistics-operations-database/delivery_events.csv',
 'raw/kaggle/yogape/logistics-operations-database/driver_monthly_metrics.csv',
 'raw/kaggle/yogape/logistics-operations-database/drivers.csv',
 'raw/kaggle/yogape/logistics-operations-database/facilities.csv',
 'raw/kaggle/yogape/logistics-operations-database/fuel_purchases.csv',
 'raw/kaggle/yogape/logistics-operations-database/loads.csv',
 'raw/kaggle/yogape/logistics-operations-database/maintenance_records.csv']

In [4]:
csv_object_names = [object_name for object_name in dataset_objects if object_name.endswith(".csv")]
csv_file_names = [Path(object_name).name for object_name in csv_object_names]
csv_file_names

['customers.csv',
 'delivery_events.csv',
 'driver_monthly_metrics.csv',
 'drivers.csv',
 'facilities.csv',
 'fuel_purchases.csv',
 'loads.csv',
 'maintenance_records.csv',
 'routes.csv',
 'safety_incidents.csv',
 'trailers.csv',
 'trips.csv',
 'truck_utilization_metrics.csv',
 'trucks.csv']

In [5]:
preferred_files = ["drivers.csv", "trips.csv", "loads.csv"]
available_files = [file_name for file_name in preferred_files if file_name in csv_file_names]

if not available_files and csv_file_names:
    available_files = csv_file_names[:2]

available_files

['drivers.csv', 'trips.csv', 'loads.csv']

In [6]:
dataframes = {}
for file_name in available_files:
    dataframe = reader.read_csv_from_dataset(file_name)
    dataframes[file_name] = dataframe
    print(f"Arquivo: {file_name} | shape={dataframe.shape}")
    display(dataframe.head())

Arquivo: drivers.csv | shape=(150, 12)


,driver_id,first_name,last_name,hire_date,termination_date,license_number,license_state,date_of_birth,home_terminal,employment_status,cdl_class,years_experience
0,DRV00001,Jennifer,Hernandez,2014-10-31,NaN,DL673510887,WA,1973-11-07,Denver,Active,A,3
1,DRV00002,William,Martin,2020-10-02,NaN,DL128955006,GA,1976-11-03,Columbus,Active,A,20
2,DRV00003,Charles,Hernandez,2021-09-21,NaN,DL523076025,NC,1970-04-06,Salt Lake City,Active,A,19
3,DRV00004,Barbara,Brown,2013-09-08,NaN,DL735540030,WA,1995-02-06,Denver,Active,A,19
4,DRV00005,Mary,Martinez,2018-12-02,NaN,DL706011277,AZ,1960-07-15,Chicago,Active,A,12


Arquivo: trips.csv | shape=(85410, 12)


,trip_id,load_id,driver_id,truck_id,trailer_id,dispatch_date,actual_distance_miles,actual_duration_hours,fuel_gallons_used,average_mpg,idle_time_hours,trip_status
0,TRIP00000001,LOAD00000001,DRV00117,TRK00035,TRL00167,2022-01-01,1314,26.2,183.8,7.15,3.5,Completed
1,TRIP00000002,LOAD00000002,DRV00141,TRK00108,TRL00082,2022-01-01,515,8.6,93.6,5.50,8.3,Completed
2,TRIP00000003,LOAD00000003,DRV00032,TRK00031,TRL00138,2022-01-01,2509,45.0,339.1,7.40,12.0,Completed
3,TRIP00000004,LOAD00000004,DRV00083,TRK00105,TRL00018,2022-01-01,717,11.1,110.3,6.50,9.6,Completed
4,TRIP00000005,LOAD00000005,DRV00044,TRK00076,TRL00054,2022-01-01,2243,35.0,328.9,6.82,11.6,Completed


Arquivo: loads.csv | shape=(85410, 12)


,load_id,customer_id,route_id,load_date,load_type,weight_lbs,pieces,revenue,fuel_surcharge,accessorial_charges,load_status,booking_type
0,LOAD00000001,CUST00183,RTE00019,2022-01-01,Dry Van,19178,13,3045.23,406.72,100,Completed,Spot
1,LOAD00000002,CUST00076,RTE00058,2022-01-01,Dry Van,27761,22,1224.48,98.61,0,Completed,Dedicated
2,LOAD00000003,CUST00027,RTE00048,2022-01-01,Refrigerated,35594,16,7171.12,792.88,0,Completed,Spot
3,LOAD00000004,CUST00088,RTE00013,2022-01-01,Refrigerated,33274,10,1308.20,141.33,50,Completed,Spot
4,LOAD00000005,CUST00185,RTE00020,2022-01-01,Dry Van,40257,10,3317.18,738.48,0,Completed,Spot


In [7]:
schema_file_name = "DATABASE_SCHEMA.txt"
if schema_file_name in [Path(object_name).name for object_name in dataset_objects]:
    schema_text = reader.read_text_from_dataset(schema_file_name)
    print(schema_text[:2000])
else:
    print("DATABASE_SCHEMA.txt não encontrado no dataset.")


LOGISTICS DATABASE SCHEMA

1. DRIVERS
   - Primary Key: driver_id
   - Contains: Driver demographics, employment history, license info
   
2. TRUCKS
   - Primary Key: truck_id
   - Contains: Fleet equipment details, acquisition info, status
   
3. TRAILERS
   - Primary Key: trailer_id
   - Contains: Trailer inventory, types, status
   
4. CUSTOMERS
   - Primary Key: customer_id
   - Contains: Customer accounts, contract types, revenue potential
   
5. FACILITIES
   - Primary Key: facility_id
   - Contains: Terminal and warehouse locations, capacity
   
6. ROUTES
   - Primary Key: route_id
   - Contains: Origin-destination pairs, distances, rate structures
   
7. LOADS
   - Primary Key: load_id
   - Foreign Keys: customer_id, route_id
   - Contains: Shipment details, revenue, booking type
   
8. TRIPS
   - Primary Key: trip_id
   - Foreign Keys: load_id, driver_id, truck_id, trailer_id
   - Contains: Actual trip performance, fuel consumption, duration
   
9. FUEL_PURCHASES
   - Primary